In [62]:
import os
import skimage
from tqdm import tqdm
import time
import cv2
import argparse
import wandb
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv

import torch
import torch.nn
from torch.optim.lr_scheduler import LambdaLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using {device}")

using cuda


In [79]:
class StafLayer(nn.Module):
    """
    See paper sec. 3.2, final paragraph, and supplement Sec. 1.5 for
    discussion of omega_0.

    If is_first=True, omega_0 is a frequency factor which simply multiplies
    the activations before the nonlinearity. Different signals may require
    different omega_0 in the first layer - this is a hyperparameter.

    If is_first=False, then the weights will be divided by omega_0 so as to
    keep the magnitude of activations constant, but boost gradients to the
    weight matrix (see supplement Sec. 1.5)
    """

    def __init__(
        self,
        in_features,
        out_features,
        bias=True,
        is_first=False,
        omega_0=30,
        scale=10.0,
        init_weights=True,
    ):
        super().__init__()

        self.nf = 5
        self.omega_0 = omega_0
        self.is_first = is_first

        self.in_features = in_features

        # Initialize weights and parameters
        self.init_params()
                
    def init_params(self):
        """
        Initializes the parameters for the sinusoidal activations.
        """
        ws = self.omega_0 * torch.rand(self.nf)
        self.ws = nn.Parameter(ws, requires_grad=True)

        # Initialize phases uniformly in the range [-π, π]
        self.phis = nn.Parameter(
            -math.pi + 2 * math.pi * torch.rand(self.nf), requires_grad=True
        )

        # Initialize scale factors based on a Laplace distribution
        diversity_y = 1 / (2 * self.nf)
        laplace_samples = torch.distributions.Laplace(0, diversity_y).sample((self.nf,))
        self.bs = nn.Parameter(torch.sign(laplace_samples) * torch.sqrt(torch.abs(laplace_samples)), requires_grad=True)

    def forward(self, input):
        return self.param_act(input)

    def param_act(self, linout):
        sinusoidal_modulation = self.bs * torch.sin(self.ws * linout.unsqueeze(-1) + self.phis)
        return sinusoidal_modulation.sum(dim=-1)

    def apply_activation(self, x):
        y = torch.zeros_like(x)
        for i in range(len(self.ws)):
            y += self.bs[i] * torch.sin((self.ws[i] * x) + self.phis[i])
        return y


class Staf(nn.Module):
    def __init__(
        self,
        in_features,
        hidden_features,
        hidden_layers,
        out_features,
        outermost_linear=True,
        first_omega_0=30,
        hidden_omega_0=30.0,
        scale=10.0,
        pos_encode=False,
        sidelength=512,
        fn_samples=None,
        use_nyquist=True,
    ):
        super().__init__()
        self.pos_encode = pos_encode
        self.staf_act = StafLayer(
            hidden_features,
            hidden_features,
            omega_0=first_omega_0,
            scale=scale,
        )

        self.net = []
        self.net.append(
            nn.Sequential(*[
                nn.Linear(
                    in_features,
                    hidden_features
                ),
                self.staf_act
            ])
        )

        for i in range(hidden_layers):
            self.net.append(
                nn.Sequential(*[
                    nn.Linear(
                        hidden_features,
                        hidden_features
                    ),
                    self.staf_act
                ])
            )

        if outermost_linear:
            dtype = torch.float
            final_linear = nn.Linear(hidden_features, out_features, dtype=dtype)

            self.net.append(final_linear)
        else:
            self.net.append(
                nn.Sequential(*[
                    nn.Linear(
                        hidden_features,
                        out_features
                    ),
                    self.staf_act
                ])
            )
        self.net = nn.Sequential(*self.net)

    def forward(self, coords):
        output = self.net(coords)

        return output

In [80]:
m = Staf(
    in_features=2,
    hidden_features=256,
    hidden_layers=3,
    out_features=1,
    first_omega_0=30.0,
    hidden_omega_0=30.0,
    scale=10.0,
    pos_encode=False,
    sidelength=512,
    fn_samples=None,
    use_nyquist=None,
)
m

Staf(
  (staf_act): StafLayer()
  (net): Sequential(
    (0): Sequential(
      (0): Linear(in_features=2, out_features=256, bias=True)
      (1): StafLayer()
    )
    (1): Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): StafLayer()
    )
    (2): Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): StafLayer()
    )
    (3): Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): StafLayer()
    )
    (4): Linear(in_features=256, out_features=1, bias=True)
  )
)

In [77]:
def get_model(
    non_linearity,
    hidden_features,
    hidden_layers,
    out_features,
    first_omega_0=30,
    hidden_omega_0=30,
    scale=10,
    sidelength=512,
    fn_samples=None,
    use_nyquist=True,
):
    """
    Function to get a class instance for a given type of
    implicit neural representation

    Inputs:
        non_linearity: One of 'parac', 'wire', 'siren'.
        in_features: Number of input features. 2 for image, 3 for volume and so on.
        hidden_features: Number of features per hidden layer
        hidden_layers: Number of hidden layers
        out_features; Number of outputs features. 3 for color image, 1 for grayscale or volume and so on
        first_omega0 (30): For siren and wire only: Omega for first layer
        hidden_omega0 (30): For siren and wire only: Omega for hidden layers
        scale (10): For wire and gauss only: Scale for Gaussian window
        pos_encode (False): If True apply positional encoding
        sidelength (512): if pos_encode is true, use this for side length parameter
        fn_samples (None): Redundant parameter
        use_nyquist (True): if True, use nyquist sampling for positional encoding

    Outputs:
        Model instance
    """

    if non_linearity == "staf":
        model = Staf(
            in_features=2,
            hidden_features=hidden_features,
            hidden_layers=hidden_layers,
            out_features=out_features,
            first_omega_0=first_omega_0,
            hidden_omega_0=hidden_omega_0,
            scale=scale,
            pos_encode=False,
            sidelength=sidelength,
            fn_samples=fn_samples,
            use_nyquist=use_nyquist,
        )

    elif non_linearity == "wire":
        model = Wire(
            in_features=2,
            hidden_features=hidden_features,
            hidden_layers=hidden_layers,
            out_features=out_features,
            first_omega_0=first_omega_0,
            hidden_omega_0=hidden_omega_0,
            scale=scale,
            pos_encode=False,
            sidelength=sidelength,
            fn_samples=fn_samples,
            use_nyquist=use_nyquist,
        )
    elif non_linearity == "siren":
        model = Siren(
            in_features=2,
            hidden_features=hidden_features,
            hidden_layers=hidden_layers,
            out_features=out_features,
            first_omega_0=first_omega_0,
            hidden_omega_0=hidden_omega_0,
            scale=scale,
            pos_encode=False,
            sidelength=sidelength,
            fn_samples=fn_samples,
            use_nyquist=use_nyquist,
        )

    elif non_linearity == 'kan':
        model = models.INR(non_linearity).run(
            in_features=2,
            out_features=out_features,
            hidden_features=hidden_features,
            hidden_layers=hidden_layers,
            degree=16,
        )


    return model

In [67]:
"""
Miscellaneous utilities that are helpful but cannot be clubbed into other modules.
"""

def normalize(x, full_normalize=False):
    """
    Normalize input to lie between 0, 1.

    Inputs:
        x: Input signal
        full_normalize: If True, normalize such that minimum is 0 and
            maximum is 1. Else, normalize such that maximum is 1 alone.

    Outputs:
        xnormalized: Normalized x.
    """

    if x.sum() == 0:
        return x

    xmax = x.max()

    if full_normalize:
        xmin = x.min()
    else:
        xmin = 0

    xnormalized = (x - xmin) / (xmax - xmin)

    return xnormalized


def psnr(x, xhat):
    """Compute Peak Signal to Noise Ratio in dB

    Inputs:
        x: Ground truth signal
        xhat: Reconstructed signal

    Outputs:
        snrval: PSNR in dB
    """
    err = x - xhat
    denom = np.mean(pow(err, 2))

    snrval = 10 * np.log10(np.max(x) / denom)

    return snrval


def count_parameters(model):
    """Count the number of paramaters.
    Inputs:
        model: Model instance

    Outputs:
        Number of parameters of the model
    """
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [88]:
def train(args, wandb_xp=None):
    """
    Train the model.

    Inputs:
        args: training arguments.
        wandb_xp: WANDB expriment instance.

    Outputs:
        model: trained model
        reconstructed image: The last reconstructed image.
    """

    if args.non_linearity == "wire":
        # Gabor filter constants.
        # We suggest omega0 = 4 and sigma0 = 4 for reconst, and omega0=20, sigma0=30 for image representation
        omega0 = 20  # Frequency of sinusoid
        sigma0 = 30  # Sigma of Gaussian

    else:
        omega0 = 30.0  # Frequency of sinusoid
        sigma0 = 4.0  # Sigma of Gaussian

    img = load_image_data(args.input_image)
    img = cv2.resize(img, None, fx=1 / 4, fy=1 / 4, interpolation=cv2.INTER_AREA)
    img = normalize(img.astype(np.float32), full_normalize=True)

    H, W = img.shape[0], img.shape[1]
    print(f'H: {H} | W: {W}')
    if len(img.shape) == 2:
        # grayscale image
        img_dim = 1
        img = img[:, :, np.newaxis]
    else:
        # rgb image
        img_dim = 3

    model = get_model(
        non_linearity=args.non_linearity,
        out_features=img_dim,
        hidden_features=256,
        hidden_layers=3,
        first_omega_0=omega0,
        hidden_omega_0=omega0,
        scale=sigma0,
    ).to(device)

    print("Number of parameters: ", count_parameters(model))

    optim = torch.optim.Adam(
        lr=args.lr, betas=(0.9, 0.999), eps=1e-08, params=model.parameters()
    )

    # Schedule to reduce lr to 0.1 times the initial rate in final epoch
    lr_sched = LambdaLR(optim, lambda x: 0.1 ** min(x / args.epochs, 1))

    X, Y = torch.meshgrid(
        torch.linspace(-1, 1, W), torch.linspace(-1, 1, H), indexing="xy"
    )
    coords = torch.hstack((X.reshape(-1, 1), Y.reshape(-1, 1))).to(device)

    gt = (
        torch.tensor(img).reshape(H * W, img_dim).to(device)
    )  # the model output will be of the shape 3 -> (x, y, z) - so ground truth also have to have shape of 3

    prog_bar = tqdm(range(args.epochs))
    psnr_vals = []

    for epoch in prog_bar:
        indices = torch.randperm(H * W).to(device)
        reconst_arr = torch.zeros_like(gt).to(device)

        # batch training
        train_loss = cnt = 0
        for start_idx in range(0, H * W, args.batch_size):
            end_idx = min(H * W, start_idx + args.batch_size)

            batch_indices = indices[start_idx:end_idx].to(device)
            batch_coords = coords[batch_indices, ...].unsqueeze(0)

            pixel_vals_preds = model(batch_coords)

            loss = ((pixel_vals_preds - gt[batch_indices, :]) ** 2).mean()
            train_loss += loss.item()

            model.zero_grad()
            loss.backward()
            optim.step()

            with torch.no_grad():
                reconst_arr[batch_indices, :] = pixel_vals_preds.squeeze(0)

            cnt += 1

        # # no batch training -> only for comparison
        # pixel_vals_preds = model(coords.unsqueeze(0))
        # loss = ((pixel_vals_preds - gt) ** 2).mean()
        # train_loss += loss.item()

        # model.zero_grad()
        # loss.backward()
        # optim.step()

        # with torch.no_grad():
        #     reconst_arr = pixel_vals_preds.squeeze(0)

        # cnt += 1
        # # no batch training -> only for comparison

        # evaluation
        with torch.no_grad():
            reconst_arr = reconst_arr.detach().cpu().numpy()
            psnr_val = psnr(gt.detach().cpu().numpy(), reconst_arr)
            psnr_vals.append(psnr_val)

            prog_bar.set_description(f"PSNR: {psnr_val:.1f} dB")
            prog_bar.refresh()

            if wandb_xp:
                wandb_xp.log({"train loss": train_loss / cnt, "psnr": psnr_val})

        lr_sched.step()

        if args.live:
            cv2.imshow("Reconstruction", reconst_arr.reshape(W, H, img_dim))
            cv2.waitKey(1)

    # np.save(
    #     os.path.join(
    #         os.path.join(os.getenv("RESULTS_SAVE_PATH", "."), "reconst"),
    #         f"{args.non_linearity}_psnr_vals.npy",
    #     ),
    #     psnr_vals,
    # )

    return model, reconst_arr.reshape(W, H, img_dim)

In [89]:
import skimage

def load_image_data(image_name):
    """
    Loading image from skimage.
    If the image name is correct, it will return the iamge.
    Otherwise, it'll raise an exception.

    Inputs:
        image_name: name of the image - for example 'camera"

    Outputs:
        image file from skimage.
    """

    if hasattr(skimage.data, image_name):
        input_image_get_fn = getattr(skimage.data, image_name)
        return input_image_get_fn()
    else:
        raise Exception(
            f'Image name is wrong. Skimage does not have image with the name "{image_name}".'
        )


In [90]:
def get_args(args_str):
    """
    Get training arguemtns.

    Outputs:
        args: arguemnts object.
    """
    parser = argparse.ArgumentParser(description="Image reconstruction parameters")
    parser.add_argument(
        "-i",
        "--input_image",
        type=str,
        help="Input image name from skimage.",
        default="camera",
    )
    parser.add_argument(
        "-n",
        "--non_linearity",
        choices=["staf", "wire", "siren", "kan"],
        type=str,
        help="Name of non linearity",
        default="staf",
    )
    parser.add_argument(
        "-e", "--epochs", type=int, help="Epcohs of maining", default=250
    )
    parser.add_argument(
        "--lr",
        type=float,
        default=1e-4,
        help="Learning rate. Parac works best at 1e-4, Wire at 5e-3 to 2e-2, SIREN at 1e-3 to 2e-3.",
    )
    parser.add_argument(
        "-b",
        "--batch_size",
        type=int,
        default=128 * 128,
        help="Batch size.",
    )
    parser.add_argument(
        "--live",
        type=bool,
        default=False,
        action=argparse.BooleanOptionalAction,
        help="Whether to show the reconstructed image live during the training or not.",
    )

    return parser.parse_args(args_str.split())

In [91]:
args = get_args('-n staf')
args

Namespace(input_image='camera', non_linearity='staf', epochs=250, lr=0.0001, batch_size=16384, live=False)

In [92]:
m, img_recont = train(args)

H: 128 | W: 128
Number of parameters:  198416


PSNR: 40.7 dB: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 250/250 [00:36<00:00,  6.79it/s]
